# Wav2Vec2-base (frozen) + SVM

**Модель:** Wav2Vec2-base (frozen) → mean+std+max → SVM RBF

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from transformers import Wav2Vec2Model, Wav2Vec2Processor

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds, get_durations, load_audio, get_durations
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_cv_fold

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)


cpu


/home/dk/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
MODEL_ID = "facebook/wav2vec2-base"
w2v_proc  = Wav2Vec2Processor.from_pretrained(MODEL_ID)
w2v_model = Wav2Vec2Model.from_pretrained(MODEL_ID).to(DEVICE)
w2v_model.eval()

def extract_wav2vec(path):
    y, _ = load_audio(path, sr=16000)
    inp = w2v_proc(y, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        hs = w2v_model(inp.input_values.to(DEVICE)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()
print(f"Train+Val: {len(paths_trainval)}  |  Test: {len(paths_test)}")

Train+Val: 2356  |  Test: 416


In [4]:

# --- Этап 1: извлечение эмбеддингов (один раз) ---
print("Извлечение признаков (train+val)...")
X_embeds_trainval = build_feature_matrix(paths_trainval, extract_wav2vec, n_jobs=1)
print("Извлечение признаков (test)...")
X_embeds_test     = build_feature_matrix(paths_test,     extract_wav2vec, n_jobs=1)

del w2v_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

path_to_idx = {p: i for i, p in enumerate(paths_trainval)}

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
}

# --- Этап 2: 5-fold CV на предвычисленных эмбеддингах ---
with start_run("exp_wav2vec_svm", group="03_pretrained_frozen"):

    mlflow.log_params({
        "encoder":        MODEL_ID,
        "encoder_frozen": True,
        "aggregation":    "mean+std+max",
        "embed_dim":      2304,
        "classifier":     "SVM RBF",
        "cv_folds":       config.CV_N_SPLITS,
        "class_weight":   "balanced",
        "note":           "embeddings precomputed once before CV",
    })

    fold_results = []
    t0 = time.perf_counter()

    for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
            get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

        print(f"\nФолд {fold+1}/{config.CV_N_SPLITS}")
        idx_tr  = [path_to_idx[p] for p in paths_tr]
        idx_val = [path_to_idx[p] for p in paths_val]
        X_tr  = np.hstack([X_embeds_trainval[idx_tr],  letters_tr,  get_durations(paths_tr).reshape(-1, 1)])
        X_val = np.hstack([X_embeds_trainval[idx_val], letters_val, get_durations(paths_val).reshape(-1, 1)])

        grid = GridSearchCV(pipeline, param_grid, cv=3,
                            scoring="f1_macro", refit=True, n_jobs=-1)
        grid.fit(X_tr, labels_tr)
        clf = grid.best_estimator_
        print(f"  best={grid.best_params_}")

        y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
        thr = find_optimal_threshold(labels_val, y_proba_val)
        metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=True)
        fold_results.append(metrics)
        log_cv_fold(fold, f1_bad=metrics["f1_bad"], f1_macro=metrics["f1_macro"],
                    roc_auc=metrics["roc_auc"], threshold=thr)

    train_time_sec = time.perf_counter() - t0
    cv_agg = evaluate_cv_folds(fold_results)
    print_cv_summary(cv_agg)
    _run_id = mlflow.active_run().info.run_id


Извлечение признаков (train+val)...
Извлечение признаков (test)...

Фолд 1/5
  best={'clf__C': 2.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.85      0.77      0.81       319
         bad       0.60      0.73      0.66       153

    accuracy                           0.76       472
   macro avg       0.73      0.75      0.73       472
weighted avg       0.77      0.76      0.76       472

Threshold : 0.38
Accuracy  : 0.7564
F1-macro  : 0.7346
F1-bad    : 0.6588
ROC-AUC   : 0.7993

Фолд 2/5
  best={'clf__C': 2.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.83      0.80      0.81       319
         bad       0.61      0.66      0.63       152

    accuracy                           0.75       471
   macro avg       0.72      0.73      0.72       471
weighted avg       0.76      0.75      0.76       471

Threshold : 0.39
Accuracy  : 0.7537
F1-macro  : 0.7238
F1-bad    : 0.6329


In [5]:
with mlflow.start_run(run_id=_run_id):

    print("\nФинальная модель на train+val...")
    X_trainval = np.hstack([X_embeds_trainval, letters_trainval, get_durations(paths_trainval).reshape(-1, 1)])
    X_test     = np.hstack([X_embeds_test,     letters_test,     get_durations(paths_test).reshape(-1, 1)])

    final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                              scoring="f1_macro", refit=True, n_jobs=-1)
    final_grid.fit(X_trainval, labels_trainval)
    mlflow.log_params({f"best_{k}": v for k, v in final_grid.best_params_.items()})

    cv_threshold = cv_agg["threshold_mean"]
    y_proba_test = final_grid.best_estimator_.predict_proba(X_test)[:, config.CLASS_BAD]
    test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)
    pd.DataFrame({
        "path":    paths_test,
        "y_true":  labels_test,
        "y_pred":  (y_proba_test >= cv_threshold).astype(int),
        "y_proba": y_proba_test,
    }).to_csv(exp_dir / "test_predictions.csv", index=False)

    save_result_csv(
        exp_dir=exp_dir, experiment_id="exp_wav2vec_svm",
        experiment_name="Wav2Vec2-base (frozen) + SVM",
        model="Wav2Vec2-base (frozen) + SVM RBF",
        accuracy=test_metrics["accuracy"], f1_macro=test_metrics["f1_macro"],
        f1_bad=test_metrics["f1_bad"],     roc_auc=test_metrics["roc_auc"],
        precision_bad=test_metrics["precision_bad"], recall_bad=test_metrics["recall_bad"],
        threshold=test_metrics["threshold"],
        cv_f1_bad_std=cv_agg["f1_bad_std"],
        cv_f1_macro_std=cv_agg["f1_macro_std"],
        cv_accuracy_std=cv_agg["accuracy_std"],
        cv_roc_auc_std=cv_agg["roc_auc_std"],
        cv_precision_bad_std=cv_agg["precision_bad_std"],
        cv_recall_bad_std=cv_agg["recall_bad_std"],
        cv_threshold_std=cv_agg["threshold_std"],
        embed_dim=2304,
        embed_dim_note="Wav2Vec2-base 768-dim × 3 (mean+std+max)",
        notes=f"5-fold CV + holdout | best={final_grid.best_params_} | thr={cv_threshold:.2f}",
        train_time_sec=train_time_sec,
    )
    print("Результаты сохранены")

df_folds = pd.DataFrame(fold_results)
df_folds.index = [f"Fold {i+1}" for i in range(len(fold_results))]
display(df_folds[["accuracy", "f1_macro", "f1_bad", "roc_auc", "threshold"]])



Финальная модель на train+val...
              precision    recall  f1-score   support

        good       0.84      0.81      0.83       281
         bad       0.64      0.69      0.66       135

    accuracy                           0.77       416
   macro avg       0.74      0.75      0.74       416
weighted avg       0.78      0.77      0.77       416

Threshold : 0.33
Accuracy  : 0.7716
F1-macro  : 0.7448
F1-bad    : 0.6619
ROC-AUC   : 0.8144
Результаты сохранены


,accuracy,f1_macro,f1_bad,roc_auc,threshold
Fold 1,0.756356,0.734649,0.658754,0.799260,0.38
Fold 2,0.753715,0.723804,0.632911,0.793908,0.39
Fold 3,0.779193,0.762463,0.699422,0.850767,0.33
Fold 4,0.726115,0.715735,0.661417,0.815295,0.24
Fold 5,0.747346,0.728530,0.657061,0.807847,0.29
